## Experiment No: 6
## Experiment Title: N-gram Language Model and Perplexity Analysis

**Name:** Himanshu Jadhav  
**Roll Number:** TE-32

### Step 1: Import Libraries

In [1]:
import math
from collections import Counter

import nltk
import pandas as pd
from tabulate import tabulate

for resource in ("punkt", "punkt_tab", "gutenberg"):
    nltk.download(resource, quiet=True)

from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize

print("Import and NLTK setup successful")

Import and NLTK setup successful


### Step 2: Load a Text Corpus (NLTK Gutenberg)

In [2]:
raw_text = gutenberg.raw('carroll-alice.txt')[:20000]
tokens   = [w.lower() for w in word_tokenize(raw_text) if w.isalpha()]
vocabulary = list(set(tokens))

print("Total Tokens in Corpus:", len(tokens))
print("Sample Tokens:", tokens[:20])
print("Unique Words in Vocabulary:", len(vocabulary))

Total Tokens in Corpus: 3704
Sample Tokens: ['alice', 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'chapter', 'i', 'down', 'the', 'alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting']
Unique Words in Vocabulary: 820


### Step 3: Train-Test Split

In [3]:
split_point  = int(len(tokens) * 0.8)
train_tokens = tokens[:split_point]
test_tokens  = tokens[split_point:]

print("Training Tokens:", len(train_tokens))
print("Testing  Tokens:", len(test_tokens))

Training Tokens: 2963
Testing  Tokens: 741


### Step 4: Generate Unigrams, Bigrams and Trigrams

In [4]:
def get_ngrams(tokens, n):
    return list(nltk.ngrams(tokens, n))

def show_top_ngrams(counter, n=5, label="Bigram"):
    df = pd.DataFrame(counter.most_common(n), columns=[label, "Count"])
    print(tabulate(df, headers='keys', tablefmt='fancy_grid', showindex=False ))

In [5]:
unigrams = get_ngrams(train_tokens, 1)
bigrams  = get_ngrams(train_tokens, 2)
trigrams = get_ngrams(train_tokens, 3)

unigram_counts = Counter(unigrams)
bigram_counts  = Counter(bigrams)
trigram_counts = Counter(trigrams)

print("Unique Unigrams :", len(unigram_counts))
print("Unique Bigrams  :", len(bigram_counts))
print("Unique Trigrams :", len(trigram_counts))

Unique Unigrams : 732
Unique Bigrams  : 2293
Unique Trigrams : 2832


In [6]:
print("\nTop 5 Unigrams:")
show_top_ngrams(unigram_counts, n=5, label="Unigram")


Top 5 Unigrams:
╒═══════════╤═════════╕
│ Unigram   │   Count │
╞═══════════╪═════════╡
│ ('the',)  │     137 │
├───────────┼─────────┤
│ ('she',)  │     110 │
├───────────┼─────────┤
│ ('to',)   │      97 │
├───────────┼─────────┤
│ ('and',)  │      97 │
├───────────┼─────────┤
│ ('it',)   │      72 │
╘═══════════╧═════════╛


In [7]:
print("\nTop 5 Bigrams:")
show_top_ngrams(bigram_counts, n=5, label="Bigram")


Top 5 Bigrams:
╒════════════════╤═════════╕
│ Bigram         │   Count │
╞════════════════╪═════════╡
│ ('she', 'was') │      16 │
├────────────────┼─────────┤
│ ('of', 'the')  │      15 │
├────────────────┼─────────┤
│ ('it', 'was')  │      12 │
├────────────────┼─────────┤
│ ('and', 'she') │      12 │
├────────────────┼─────────┤
│ ('in', 'the')  │       9 │
╘════════════════╧═════════╛


In [8]:
print("\nTop 5 Trigrams:")
show_top_ngrams(trigram_counts, n=5, label="Trigram")


Top 5 Trigrams:
╒════════════════════════╤═════════╕
│ Trigram                │   Count │
╞════════════════════════╪═════════╡
│ ('one', 'of', 'the')   │       4 │
├────────────────────────┼─────────┤
│ ('i', 'must', 'be')    │       4 │
├────────────────────────┼─────────┤
│ ('that', 'she', 'was') │       4 │
├────────────────────────┼─────────┤
│ ('she', 'went', 'on')  │       4 │
├────────────────────────┼─────────┤
│ ('as', 'she', 'could') │       3 │
╘════════════════════════╧═════════╛


### Step 5: Vocabulary Size (needed for smoothing)

In [9]:
vocabulary_size = len(vocabulary)
print("Vocabulary Size:", vocabulary_size)

Vocabulary Size: 820


### Step 6: Bigram Probability with Add-One (Laplace) Smoothing

In [10]:
def ngram_prob(ngram, counts, lower_counts, vocab_size, smoothing=True):
    ngram = tuple(ngram)
    ngram_count = counts.get(ngram, 0)

    # For an n-gram, the conditioning context is all tokens except the last.
    prefix = ngram[:-1]
    prefix_count = lower_counts.get(prefix, 0)

    if smoothing:
        # Add-one smoothing
        prob = (ngram_count + 1) / (prefix_count + vocab_size)
    else:
        prob = ngram_count / prefix_count if prefix_count > 0 else 0.0

    log_prob = math.log(prob) if prob > 0 else float("-inf")
    return prob, log_prob


# Use training vocabulary for smoothing to avoid test-set information leakage.
vocabulary = sorted(set(train_tokens))
vocabulary_size = len(vocabulary)

print(f"Vocabulary Size (training set): {vocabulary_size}")

# Example probabilities
p_uni, log_uni = ngram_prob(
    ("alice",), unigram_counts,
    Counter({(): len(train_tokens)}),
    vocabulary_size
)
print(f"P('alice') with smoothing = {p_uni:.6f} (log: {log_uni:.4f})")

p_bi, log_bi = ngram_prob(
    ("alice", "was"),
    bigram_counts,
    unigram_counts,
    vocabulary_size
)
print(f"P('was' | 'alice') with smoothing = {p_bi:.6f} (log: {log_bi:.4f})")

p_tri, log_tri = ngram_prob(
    ("alice", "was", "beginning"),
    trigram_counts,
    bigram_counts,
    vocabulary_size
)
print(f"P('beginning' | 'alice was') with smoothing = {p_tri:.6f} (log: {log_tri:.4f})")


Vocabulary Size (training set): 732
P('alice') with smoothing = 0.010555 (log: -4.5512)
P('was' | 'alice') with smoothing = 0.005195 (log: -5.2601)
P('beginning' | 'alice was') with smoothing = 0.002721 (log: -5.9067)


### Step 7: Perplexity Function

In [11]:
def calculate_perplexity(test_tokens, n, smoothing=True):
    if n not in (1, 2, 3):
        raise ValueError("n must be 1, 2, or 3.")

    test_ngrams = get_ngrams(test_tokens, n)
    if not test_ngrams:
        raise ValueError("The test set is too small for the selected n.")

    log_prob_sum = 0.0

    for ngram in test_ngrams:
        if n == 1:
            prob, _ = ngram_prob(
                ngram,
                unigram_counts,
                Counter({(): len(train_tokens)}),
                vocabulary_size,
                smoothing=smoothing
            )
        elif n == 2:
            prob, _ = ngram_prob(
                ngram,
                bigram_counts,
                unigram_counts,
                vocabulary_size,
                smoothing=smoothing
            )
        else:
            prob, _ = ngram_prob(
                ngram,
                trigram_counts,
                bigram_counts,
                vocabulary_size,
                smoothing=smoothing
            )

        log_prob_sum += math.log(prob)

    return math.exp(-log_prob_sum / len(test_ngrams))


### Step 8: Compute and Compare Perplexity for Unigram, Bigram, Trigram

In [12]:
unigram_perplexity = calculate_perplexity(test_tokens, 1)
bigram_perplexity  = calculate_perplexity(test_tokens, 2)
trigram_perplexity = calculate_perplexity(test_tokens, 3)

print(f"Unigram Perplexity: {unigram_perplexity:.4f}")
print(f"Bigram  Perplexity: {bigram_perplexity:.4f}")
print(f"Trigram Perplexity: {trigram_perplexity:.4f}")

Unigram Perplexity: 317.0716
Bigram  Perplexity: 490.2204
Trigram Perplexity: 690.2056


### Final Output

In [13]:
result_table = pd.DataFrame({
    "Model": ["Unigram", "Bigram", "Trigram"],
    "Perplexity": [
        round(unigram_perplexity, 2),
        round(bigram_perplexity, 2),
        round(trigram_perplexity, 2)
    ]
})

print("Experiment Completed Successfully")
print()
print(tabulate(result_table, headers="keys", tablefmt="psql", showindex=False))


Experiment Completed Successfully

+---------+--------------+
| Model   |   Perplexity |
|---------+--------------|
| Unigram |       317.07 |
| Bigram  |       490.22 |
| Trigram |       690.21 |
+---------+--------------+


### Interpretation

Lower perplexity indicates that the model assigns higher probability to the test sequence. In this experiment, compare the three values to observe how adding context from bigrams and trigrams affects language-model performance.
